# Fruit Freshness Inspector — Reverse Logistics Agent

A simplified multi-agent Gradio app for inspecting **returned fruit shipments** in a food
reverse-logistics flow. A trained image classifier (MobileNetV2) labels each fruit photo
**Fresh** or **Stale**; a deterministic rule converts that into **PASS / REJECT**; and a
small LangGraph multi-agent layer (powered by Gemini's free tier) explains results, reports
batch QC stats, and suggests handling steps — grounded in the real classification output.

**Dataset:** [Fresh and Stale Images of Fruits and Vegetables](https://www.kaggle.com/datasets/sriramr24/fresh-and-stale-images-of-fruits-and-vegetables)
We train a small binary classifier on it instead of asking a vision LLM zero-shot.

**How this differs from the original casting-defect notebook:**
- Fewer source files: `config`, `db`, `vision_tools`, `agents` (RAG + routing + all agents), `graph`, `app`
- No separate `router.py`, `rag.py`, or per-agent files — all merged into `agents.py`
- No conversation history persistence — stateless per query
- Simpler LangGraph (6 nodes, no DB-save node)
- Cleaner Gradio UI

**Limitation:** Like any trained classifier, this only generalises to images similar to its
training distribution. For other produce types, retrain Part 1 on a matching dataset.


In [ ]:
# 1. Create working directory
import os
os.makedirs('/content/fruit_inspector/data/uploads', exist_ok=True)
os.makedirs('/content/fruit_inspector/data/models', exist_ok=True)
%cd /content/fruit_inspector

## Part 1 — Download dataset and train a freshness classifier

GPU is recommended (Runtime → Change runtime type → GPU) but not required —
the MobileNetV2 backbone is frozen; only a small dense head is trained.


In [ ]:
# 2. Download the fruit freshness dataset from Kaggle
# If this exact slug fails, search Kaggle for 'fresh stale fruit' and substitute the correct one.
!pip install -q kagglehub
import kagglehub

dataset_path = kagglehub.dataset_download('raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables')
print('Dataset path:', dataset_path)

In [ ]:
# 3. Locate train/ and test/ directories (works regardless of kagglehub nesting)
import glob, os

def find_dir(root, name):
    matches = [m for m in glob.glob(os.path.join(root, '**', name), recursive=True) if os.path.isdir(m)]
    if not matches:
        raise FileNotFoundError(f'Could not find a {name!r} directory under {root}')
    return sorted(matches, key=len)[0]

train_dir = find_dir(dataset_path, 'train')
test_dir  = find_dir(dataset_path, 'test')
print('train_dir:', train_dir)
print('test_dir :', test_dir)

for split in (train_dir, test_dir):
    for sub in sorted(os.listdir(split)):
        p = os.path.join(split, sub)
        if os.path.isdir(p):
            print(f'  {split}/{sub}: {len(os.listdir(p))} images')

In [ ]:
# 4. Peek at sample images from each class
import matplotlib.pyplot as plt
from PIL import Image

classes = [c for c in sorted(os.listdir(train_dir)) if os.path.isdir(os.path.join(train_dir, c))]
fig, axes = plt.subplots(len(classes), 4, figsize=(14, 4 * len(classes)))
if len(classes) == 1:
    axes = [axes]
for row, cls in enumerate(classes):
    cls_path = os.path.join(train_dir, cls)
    files = sorted(os.listdir(cls_path))[:4]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(cls_path, fname))
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls, fontsize=10)
        axes[row][col].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Build train / validation / test datasets
import tensorflow as tf

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

class_names = train_ds.class_names
print('Class names (alphabetical = label indices 0, 1, ...):', class_names)

In [ ]:
# 6. Build model: frozen MobileNetV2 backbone + small trainable head
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs  = layers.Input(shape=IMG_SIZE + (3,))
x       = layers.Rescaling(scale=1./127.5, offset=-1)(inputs)   # [0,255] -> [-1,1]
x       = base(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 7. Train (only the head; backbone stays frozen)
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
# 8. Plot training curves and evaluate on the held-out test split
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ('accuracy', 'loss')):
    ax.plot(history.history[metric], label='train')
    ax.plot(history.history['val_' + metric], label='val')
    ax.set_title(metric.capitalize())
    ax.legend()
plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(test_ds)
print(f'Test loss: {test_loss:.4f}  |  Test accuracy: {test_acc:.4f}')

## Part 2 — Define the label mapping (after training, not before)

`image_dataset_from_directory` assigns label indices from folder names in alphabetical order.
We inspect those names here and map them to domain labels (Fresh / Stale) and the
PASS / REJECT decision rule.


In [ ]:
# 9. Auto-detect which folder = fresh, which = stale; adjust manually if needed
print('Keras class names (index 0, 1, ...):', class_names)

FRESH_KEYWORDS = {'fresh', 'good', 'ok', 'ripe', 'normal', 'healthy'}
STALE_KEYWORDS = {'stale', 'rotten', 'bad', 'spoiled', 'defective', 'damaged', 'old'}

fresh_cls = stale_cls = None
for n in class_names:
    nl = n.lower()
    if any(k in nl for k in FRESH_KEYWORDS):
        fresh_cls = n
    elif any(k in nl for k in STALE_KEYWORDS):
        stale_cls = n

if not (fresh_cls and stale_cls):
    print('Auto-detection failed. Edit fresh_cls / stale_cls manually below.')
    fresh_cls = class_names[0]   # <-- change if wrong
    stale_cls = class_names[1]   # <-- change if wrong

print(f'Mapping: {fresh_cls!r} -> Fresh, {stale_cls!r} -> Stale')

LABEL_MAP = {
    fresh_cls: 'Fresh',
    stale_cls: 'Stale',
}
DECISION_RULE = {
    'Fresh': {'decision': 'PASS',   'action': 'Re-package and Distribute'},
    'Stale': {'decision': 'REJECT', 'action': 'Compost or Discard'},
}

print('LABEL_MAP    :', LABEL_MAP)
print('DECISION_RULE:', DECISION_RULE)

In [ ]:
# 10. Save the trained model and label mapping
import json, os

os.makedirs('data/models', exist_ok=True)
model.save('data/models/freshness_classifier.keras')

with open('data/models/label_map.json', 'w') as f:
    json.dump({
        'class_names':  class_names,
        'label_map':    LABEL_MAP,
        'decision_rule': DECISION_RULE,
        'image_size':   list(IMG_SIZE),
    }, f, indent=2)

print('Saved model and label map to data/models/')

## Part 3 — Embedded Gradio App

All source files are written to disk by the cells below — no `git clone` required.
Image classification uses the trained model above; Gemini's free tier is used only for
the text-reasoning agents (explain, report, guidance) and RAG embeddings.

**File layout:**
```
config.py        — env vars and paths
db.py            — SQLite inspection records
vision_tools.py  — CNN inference + batch classification
agents.py        — knowledge base (RAG), intent router, and all three agents
graph.py         — LangGraph workflow
app.py           — Gradio UI
```


In [ ]:
%%writefile requirements.txt
langchain>=0.3.0,<0.4.0
langchain-core>=0.3.0,<0.4.0
langchain-community>=0.3.0,<0.4.0
langchain-openai>=0.2.0,<0.3.0
langgraph>=0.2.0,<0.3.0
openai>=1.45.0,<2.0.0
faiss-cpu>=1.8.0,<2.0.0
gradio
python-dotenv>=1.0.0
pydantic>=2.9.0,<3.0.0
pandas>=2.0.0
numpy>=1.26.0,<2.0.0
Pillow>=10.0.0

In [ ]:
# 11. Install app dependencies (pydantic<2.11 avoids a Gradio schema bug)
!pip install -q -r requirements.txt
!pip install -q 'pydantic<2.11'

## Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click **Create API key** and copy it.
3. Run the next cell **on its own** (not via Run All) and wait for the input prompt.


In [ ]:
# 12. Save API key to .env
import getpass

while True:
    key = getpass.getpass('Enter your Google AI Studio API key: ').strip()
    if key:
        break
    print('Empty input — try again.')

with open('.env', 'w') as f:
    f.write(f'GOOGLE_API_KEY={key}\n')

print(f'Saved .env (key length: {len(key)})')

In [ ]:
# 13. Verify the key works
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_c = OpenAI(
    api_key=os.getenv('GOOGLE_API_KEY'),
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
)
try:
    r = _c.chat.completions.create(
        model='gemini-2.5-flash',
        messages=[{'role': 'user', 'content': 'Say OK'}],
    )
    print('Key works:', r.choices[0].message.content)
except Exception as e:
    print('Key test failed:', e)
    print('Re-run the previous cell and paste the key again.')

## Write application source files


In [ ]:
%%writefile config.py
'''Configuration — paths and model settings.'''
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not found in environment')
GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'

LLM_MODEL       = 'gemini-2.5-flash'
EMBEDDING_MODEL = 'gemini-embedding-001'
TEMPERATURE     = 0.3

DB_PATH       = Path('data/freshness.db')
UPLOAD_DIR    = Path('data/uploads')
MODEL_PATH    = Path('data/models/freshness_classifier.keras')
LABEL_MAP_PATH = Path('data/models/label_map.json')

DB_PATH.parent.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)

In [ ]:
%%writefile db.py
'''Lightweight SQLite store for per-image inspection records.'''
import sqlite3, json, threading
from typing import List, Dict, Any, Optional
from contextlib import contextmanager
from config import DB_PATH

_local = threading.local()

def _conn():
    if not hasattr(_local, 'c'):
        _local.c = sqlite3.connect(str(DB_PATH), timeout=30, check_same_thread=False)
        _local.c.row_factory = sqlite3.Row
    return _local.c

@contextmanager
def _cur():
    c = _conn().cursor()
    try:
        yield c
        _conn().commit()
    except Exception:
        _conn().rollback()
        raise
    finally:
        c.close()

def init_db():
    with _cur() as c:
        c.execute('''CREATE TABLE IF NOT EXISTS inspections (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            batch_id TEXT, filename TEXT, label TEXT,
            confidence REAL, decision TEXT, action TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )''')
        c.execute('CREATE INDEX IF NOT EXISTS idx_batch ON inspections(batch_id)')

def save_record(batch_id, filename, label, confidence, decision, action):
    with _cur() as c:
        c.execute(
            'INSERT INTO inspections (batch_id,filename,label,confidence,decision,action) VALUES (?,?,?,?,?,?)',
            (batch_id, filename, label, round(confidence, 4), decision, action),
        )

def get_batch(batch_id: str) -> List[Dict]:
    with _cur() as c:
        c.execute('SELECT * FROM inspections WHERE batch_id=? ORDER BY id', (batch_id,))
        return [dict(r) for r in c.fetchall()]

def latest_batch_id() -> Optional[str]:
    with _cur() as c:
        c.execute('SELECT batch_id FROM inspections ORDER BY timestamp DESC LIMIT 1')
        r = c.fetchone()
        return r['batch_id'] if r else None

def batch_summary(batch_id: str) -> Dict:
    records = get_batch(batch_id)
    if not records:
        return {'batch_id': batch_id, 'n': 0, 'message': 'No inspections found.'}
    n = len(records)
    n_pass = sum(1 for r in records if r['decision'] == 'PASS')
    label_counts: Dict[str, int] = {}
    for r in records:
        label_counts[r['label']] = label_counts.get(r['label'], 0) + 1
    return {
        'batch_id': batch_id,
        'n': n,
        'n_pass': n_pass,
        'n_reject': n - n_pass,
        'reject_rate_pct': round((n - n_pass) / n * 100, 1),
        'label_counts': label_counts,
    }

init_db()

if __name__ == '__main__':
    print('DB initialized at', DB_PATH)

In [ ]:
%%writefile vision_tools.py
'''Load the trained freshness classifier and run batch inference.'''
import json, uuid
import numpy as np
import tensorflow as tf
from pathlib import Path
from typing import List, Dict, Any, Optional
from config import MODEL_PATH, LABEL_MAP_PATH
import db

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found at {MODEL_PATH}. Run the training cells first.'
    )

_model = tf.keras.models.load_model(MODEL_PATH)
with open(LABEL_MAP_PATH) as f:
    _meta = json.load(f)

CLASS_NAMES   = _meta['class_names']
LABEL_MAP     = _meta['label_map']
DECISION_RULE = _meta['decision_rule']
IMG_SIZE      = tuple(_meta['image_size'])


def _preprocess(path: str) -> np.ndarray:
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    return np.expand_dims(tf.keras.utils.img_to_array(img), 0)


def classify_image(path: str) -> Dict[str, Any]:
    p1 = float(_model.predict(_preprocess(path), verbose=0)[0][0])
    if p1 >= 0.5:
        raw, conf = CLASS_NAMES[1], p1
    else:
        raw, conf = CLASS_NAMES[0], 1.0 - p1
    label = LABEL_MAP.get(raw, raw)
    rule  = DECISION_RULE[label]
    return {
        'filename':  Path(path).name,
        'raw_label': raw,
        'label':     label,
        'confidence': round(conf, 4),
        'decision':  rule['decision'],
        'action':    rule['action'],
    }


def classify_batch(paths: List[str], batch_id: Optional[str] = None) -> Dict[str, Any]:
    batch_id = batch_id or str(uuid.uuid4())
    records = []
    for p in paths:
        r = classify_image(p)
        db.save_record(batch_id, r['filename'], r['label'], r['confidence'], r['decision'], r['action'])
        records.append({**r, 'batch_id': batch_id})
    return {'batch_id': batch_id, 'records': records}


if __name__ == '__main__':
    print('CLASS_NAMES :', CLASS_NAMES)
    print('LABEL_MAP   :', LABEL_MAP)
    print('DECISION_RULE:', DECISION_RULE)

In [ ]:
%%writefile agents.py
'''Knowledge base (RAG), intent router, and all three specialist agents.'''
import os, json
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from config import GOOGLE_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE, EMBEDDING_MODEL
import db

client = OpenAI(api_key=GOOGLE_API_KEY, base_url=GEMINI_BASE_URL)
_emb = OpenAIEmbeddings(
    model=EMBEDDING_MODEL, api_key=GOOGLE_API_KEY,
    base_url=GEMINI_BASE_URL, check_embedding_ctx_length=False,
)

# ── Knowledge base ─────────────────────────────────────────────────────────
_DOCS = [
    Document(
        page_content=(
            'Freshness Inspection Process: '
            'Returned or incoming fruit shipments are photographed and run through a trained classifier. '
            'Each image is labelled Fresh or Stale. '
            'Fresh items are re-packaged and sent to distribution. '
            'Stale items are rejected and routed to composting or disposal. '
            'The classifier only distinguishes Fresh vs Stale and does not identify the specific cause of spoilage.'
        ),
        metadata={'source': 'process'},
    ),
    Document(
        page_content=(
            'Pass / Reject Decision Rule: '
            'Fresh -> PASS: fruit continues to the packaging and distribution line. '
            'Stale -> REJECT: fruit is removed and logged for composting or supplier credit. '
            'The decision is enforced in code using label_map.json and is fully deterministic. '
            'No LLM is involved in the PASS or REJECT call.'
        ),
        metadata={'source': 'decision_rule'},
    ),
    Document(
        page_content=(
            'Common Causes of Staleness (manual follow-up needed): '
            'Over-ripening: soft spots, discolouration, or shrivelling from cell-wall breakdown. '
            'Mould or fungal growth: fuzzy or powdery surface growth. '
            'Bruising: impact damage during transit causes accelerated browning and decay. '
            'Dehydration: moisture loss gives a wrinkled or sunken appearance. '
            'The classifier flags all of these as Stale. '
            'A human must identify the specific cause before deciding whether to salvage or fully discard.'
        ),
        metadata={'source': 'spoilage_causes'},
    ),
    Document(
        page_content=(
            'QC Reporting Guidelines: '
            'A batch report must include total items inspected, PASS count, REJECT count, and reject rate percent. '
            'A reject rate above 20 percent in a single batch warrants a supplier escalation or storage review. '
            'Always reference the batch ID when filing a supplier claim for traceability.'
        ),
        metadata={'source': 'qc_reporting'},
    ),
]

_INDEX_DIR = 'data/faiss_index'
_splitter  = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)

def _build_index():
    chunks = _splitter.split_documents(_DOCS)
    vs = FAISS.from_documents(chunks, _emb)
    os.makedirs(_INDEX_DIR, exist_ok=True)
    vs.save_local(_INDEX_DIR)
    return vs

def _load_index():
    if os.path.exists(_INDEX_DIR):
        try:
            return FAISS.load_local(_INDEX_DIR, _emb, allow_dangerous_deserialization=True)
        except Exception:
            pass
    return _build_index()

_vs = _load_index()
print('Knowledge base ready')


def retrieve(query: str, k: int = 3) -> str:
    docs = _vs.similarity_search(query, k=k)
    nl = chr(10)
    return (nl + nl).join(
        '[' + d.metadata.get('source', 'doc') + '] ' + d.page_content for d in docs
    )


# ── Intent router ───────────────────────────────────────────────────────────
def classify_intent(query: str) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.1,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': (
                'Classify the user query into exactly one of these intents: '
                'inspect_explain (why a specific item passed or failed), '
                'qc_report (batch aggregate stats such as counts or reject rate), '
                'repair_guidance (what to do with rejected or stale items), '
                'general (everything else). '
                'Respond with a JSON object containing a single key called intent.'
            )},
            {'role': 'user', 'content': query},
        ],
    )
    try:
        return json.loads(resp.choices[0].message.content).get('intent', 'general')
    except Exception:
        return 'general'


# ── Shared LLM helper ───────────────────────────────────────────────────────
def _chat(system: str, user: str) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=TEMPERATURE,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
    )
    return resp.choices[0].message.content


# ── Agents ──────────────────────────────────────────────────────────────────
def inspect_explain_agent(query: str, batch_id: str = None, context: str = '') -> str:
    records = db.get_batch(batch_id or db.latest_batch_id() or '')
    nl = chr(10)
    return _chat(
        'You are a fruit freshness QC expert. '
        'Explain inspection outcomes using only the real classifier results provided. '
        'Never invent findings. Always quote the confidence score.',
        'Inspection records:' + nl + str(records) + nl + nl +
        'Context:' + nl + context + nl + nl +
        'Question: ' + query,
    )


def qc_report_agent(query: str, batch_id: str = None, context: str = '') -> str:
    summary = db.batch_summary(batch_id or db.latest_batch_id() or '')
    nl = chr(10)
    return _chat(
        'You are a QC reporting analyst. '
        'Summarise the batch using only the real stats provided. '
        'Never invent numbers. Flag if the reject rate exceeds 20 percent.',
        'Batch summary:' + nl + str(summary) + nl + nl +
        'Context:' + nl + context + nl + nl +
        'Question: ' + query,
    )


def repair_guidance_agent(query: str, batch_id: str = None, context: str = '') -> str:
    records  = db.get_batch(batch_id or db.latest_batch_id() or '')
    rejected = [r for r in records if r['decision'] == 'REJECT']
    nl = chr(10)
    return _chat(
        'You are a food quality advisor. '
        'Recommend handling steps for rejected fruit batches. '
        'Note that the classifier only outputs Stale; a human must identify the spoilage subtype '
        '(mould, bruising, dehydration, over-ripening) before deciding salvage vs. discard. '
        'If nothing was rejected in this batch, say so clearly.',
        'Rejected items:' + nl + str(rejected) + nl + nl +
        'Context:' + nl + context + nl + nl +
        'Question: ' + query,
    )


def general_agent(query: str, context: str = '') -> str:
    nl = chr(10)
    return _chat(
        'You are a helpful assistant for a fruit freshness inspection system. '
        'Answer general questions and guide the user toward specific queries.',
        'Context:' + nl + context + nl + nl + 'Question: ' + query,
    )


if __name__ == '__main__':
    print('agents.py loaded — knowledge base index built or loaded')

In [ ]:
%%writefile graph.py
'''LangGraph workflow for the Fruit Freshness Inspection Assistant.'''
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict
from agents import (
    classify_intent, retrieve,
    inspect_explain_agent, qc_report_agent, repair_guidance_agent, general_agent,
)


class State(TypedDict):
    query:    str
    batch_id: str
    intent:   str
    context:  str
    response: str
    agent:    str


def router_node(s: State) -> State:
    s['intent'] = classify_intent(s['query'])
    return s

def rag_node(s: State) -> State:
    s['context'] = retrieve(s['query'])
    return s

def inspect_node(s: State) -> State:
    s['response'] = inspect_explain_agent(s['query'], s.get('batch_id'), s['context'])
    s['agent'] = 'inspect_explain'
    return s

def qc_node(s: State) -> State:
    s['response'] = qc_report_agent(s['query'], s.get('batch_id'), s['context'])
    s['agent'] = 'qc_report'
    return s

def repair_node(s: State) -> State:
    s['response'] = repair_guidance_agent(s['query'], s.get('batch_id'), s['context'])
    s['agent'] = 'repair_guidance'
    return s

def general_node(s: State) -> State:
    s['response'] = general_agent(s['query'], s['context'])
    s['agent'] = 'general'
    return s


_ROUTE = {
    'inspect_explain': 'inspect',
    'qc_report':       'qc',
    'repair_guidance': 'repair',
}

def _route(s: State) -> Literal['inspect', 'qc', 'repair', 'general']:
    return _ROUTE.get(s['intent'], 'general')


def build_graph():
    g = StateGraph(State)
    for name, fn in [
        ('router',  router_node),
        ('rag',     rag_node),
        ('inspect', inspect_node),
        ('qc',      qc_node),
        ('repair',  repair_node),
        ('general', general_node),
    ]:
        g.add_node(name, fn)
    g.set_entry_point('router')
    g.add_edge('router', 'rag')
    g.add_conditional_edges(
        'rag', _route,
        {'inspect': 'inspect', 'qc': 'qc', 'repair': 'repair', 'general': 'general'},
    )
    for name in ('inspect', 'qc', 'repair', 'general'):
        g.add_edge(name, END)
    return g.compile()


graph = build_graph()
print('Graph compiled')


def ask(query: str, batch_id: str = None) -> Dict[str, Any]:
    return graph.invoke({
        'query': query, 'batch_id': batch_id,
        'intent': '', 'context': '', 'response': '', 'agent': '',
    })


if __name__ == '__main__':
    print(graph.get_graph().draw_ascii())
    result = ask('What does this system do?')
    print(result['response'])

In [ ]:
%%writefile app.py
'''Fruit Freshness Inspection Gradio App — run with: python app.py'''
import uuid
import pandas as pd
import gradio as gr
from vision_tools import classify_batch
from graph import ask


def run_inspection(files, batch_state):
    if not files:
        return None, pd.DataFrame(), batch_state, 'Upload at least one image first.'
    paths    = [f.name if hasattr(f, 'name') else f for f in files]
    batch_id = str(uuid.uuid4())
    result   = classify_batch(paths, batch_id)

    gallery, rows = [], []
    for r in result['records']:
        caption = r['decision'] + ' - ' + r['label'] + ' (' + f"{r['confidence']:.0%}" + ')'
        match   = next((p for p in paths if p.endswith(r['filename'])), paths[0])
        gallery.append((match, caption))
        rows.append({k: r[k] for k in ('filename', 'label', 'confidence', 'decision', 'action')})

    n      = len(result['records'])
    n_pass = sum(1 for r in result['records'] if r['decision'] == 'PASS')
    status = (
        'Batch ' + batch_id[:8] + ': '
        + str(n) + ' items inspected, '
        + str(n_pass) + ' PASS, '
        + str(n - n_pass) + ' REJECT'
    )
    return gallery, pd.DataFrame(rows), batch_id, status


def chat(query, history, batch_id):
    if not query.strip():
        return '', history or []
    result   = ask(query, batch_id)
    response = '[' + result.get('agent', '?') + '] ' + result['response']
    history  = (history or []) + [
        {'role': 'user',      'content': query},
        {'role': 'assistant', 'content': response},
    ]
    return '', history


with gr.Blocks(title='Fruit Freshness Inspector', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# Fruit Freshness Inspector' + chr(10) +
        'Upload fruit photos, run the freshness classifier, '
        'then ask the assistant about results or handling guidance.'
    )
    batch_state = gr.State(None)

    with gr.Row():
        with gr.Column():
            gr.Markdown('### 1. Upload & Inspect')
            files   = gr.File(label='Fruit images', file_count='multiple', file_types=['image'])
            run_btn = gr.Button('Run Inspection', variant='primary')
            status  = gr.Markdown()
            gallery = gr.Gallery(label='Results', columns=3, height=280)
            table   = gr.Dataframe(
                headers=['filename', 'label', 'confidence', 'decision', 'action']
            )

        with gr.Column():
            gr.Markdown('### 2. Ask the Assistant')
            chatbot = gr.Chatbot(height=380, type='messages')
            msg     = gr.Textbox(label='Question', placeholder='e.g. How many items passed?')
            with gr.Row():
                send  = gr.Button('Send', variant='primary')
                clear = gr.Button('Clear')
            gr.Examples(
                examples=[
                    ['Give me a summary of this batch results.'],
                    ['Why was an item flagged Stale?'],
                    ['What should we do with the rejected fruit?'],
                    ['How does the classifier decide Fresh vs Stale?'],
                ],
                inputs=msg,
            )

    run_btn.click(run_inspection, [files, batch_state], [gallery, table, batch_state, status])
    send.click(chat,   [msg, chatbot, batch_state], [msg, chatbot])
    msg.submit(chat,   [msg, chatbot, batch_state], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)


if __name__ == '__main__':
    demo.launch(share=False, server_name='127.0.0.1', server_port=7860)

In [ ]:
# 14. Initialize DB and build FAISS knowledge-base index
!python db.py
!python agents.py

## (Optional) Smoke test on held-out images

Runs the trained classifier on a few test images you never trained on,
as a quick sanity check before launching the UI.


In [ ]:
# 15. Smoke test — classify a handful of held-out test images
import os, sys
sys.path.insert(0, '.')
from vision_tools import classify_batch

test_paths = []
for sub in sorted(os.listdir(test_dir))[:2]:
    sub_path = os.path.join(test_dir, sub)
    if os.path.isdir(sub_path):
        files = sorted(os.listdir(sub_path))[:3]
        test_paths.extend(os.path.join(sub_path, f) for f in files)

if test_paths:
    result = classify_batch(test_paths)
    for r in result['records']:
        print(r['filename'], '->', r['decision'], '(' + r['label'] + ', conf=' + str(r['confidence']) + ')')
else:
    print('No test images found — make sure test_dir from Part 1 is still set.')

In [ ]:
# 16. Launch the Gradio app
# share=True creates a public *.gradio.live link (required on Colab)
from app import demo

demo.launch(share=True)

## Notes

- **Trained, not zero-shot:** detection comes from the MobileNetV2 classifier trained in Part 1,
  not a vision-LLM call — no per-image API cost or rate limit.
- **Label mapping defined after training:** `data/models/label_map.json` records the raw
  folder-derived class names alongside Fresh/Stale labels and the PASS/REJECT rule.
  Edit that file (not the model) to change the decision policy.
- **Generalisation limit:** the classifier only works well on images resembling the training
  distribution. To inspect other produce types, retrain Part 1 on a matching dataset.
- **Rate limits:** Gemini free tier caps `gemini-2.5-flash` at 15 requests/minute and
  `gemini-embedding-001` at 5 requests/minute. For the initial FAISS index build you may
  hit the embedding limit — if so, wait 60 s and re-run cell 14.
- **Restarting:** if you restart the Colab runtime, re-run all cells from the top —
  the working directory, trained model, `.env`, database, and FAISS index are all wiped.
- **Stopping the app:** use Runtime → Interrupt execution.
